
## Cue set summary

This notebook walks through the cue set used for each session per mouse. Cue set labels are **not stored in the HDF5 outputs**, so we read them from the original JSON files referenced by each HDF5 `metadata.source_file`. The rest of the information (animal, date, session) comes from the HDF5 metadata itself.


In [1]:

from pathlib import Path
import pandas as pd
import re
from datetime import datetime


In [2]:

# Root folder containing per-session HDF5 files
HDF5_ROOT = Path("/groups/spruston/home/moharb/DELTA_Behavior/outputs/hdf5")
print(HDF5_ROOT)
assert HDF5_ROOT.exists(), "HDF5 output folder not found"
print(f"Scanning HDF5 files under: {HDF5_ROOT.resolve()}")


/groups/spruston/home/moharb/DELTA_Behavior/outputs/hdf5
Scanning HDF5 files under: /groups/spruston/home/moharb/DELTA_Behavior/outputs/hdf5


In [3]:

# Helper to stream a JSON log and find the first occurrence of "cueSet"
cue_pattern = re.compile(r'"cueSet"\s*:\s*"([^"]+)"')

def extract_cue_set(json_path: Path, chunk_size: int = 1024 * 1024):
    buffer = ""
    try:
        with json_path.open("r", encoding="utf-8") as f:
            for chunk in iter(lambda: f.read(chunk_size), ""):
                buffer += chunk
                match = cue_pattern.search(buffer)
                if match:
                    return match.group(1)
                buffer = buffer[-100:]
    except FileNotFoundError:
        return None
    return None


In [4]:

# Build a per-session table from HDF5 metadata, augment with cue set from the source JSON
session_records = []
pattern = re.compile(r"Log\s+(BM\d+)\s+(\d{4}-\d{2}-\d{2})\s+session\s+(\d+)", re.IGNORECASE)

for h5_path in sorted(HDF5_ROOT.rglob("*.h5")):
    try:
        md = pd.read_hdf(h5_path, "metadata")
    except Exception as exc:
        print(f"Skipping {h5_path}: {exc}")
        continue

    source_file = Path(str(md.get("source_file", "")))
    match = pattern.search(source_file.stem)
    if not match:
        continue

    animal = match.group(1).upper()
    date = datetime.strptime(match.group(2), "%Y-%m-%d").date()
    session = int(match.group(3))
    cue_set = extract_cue_set(source_file)

    session_records.append({
        "animal": animal,
        "date": date,
        "session": session,
        "cue_set": cue_set,
        "h5_path": str(h5_path),
        "json_path": str(source_file),
    })

sessions = pd.DataFrame(session_records).sort_values(["animal", "date", "session"])
sessions


,animal,date,session,cue_set,h5_path,json_path
0,BM32,2025-06-16,1,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM32/Log BM3...
26,BM32,2025-06-16,1,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM32/Log BM3...
1,BM32,2025-06-16,2,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM32/Log BM3...
27,BM32,2025-06-16,2,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM32/Log BM3...
2,BM32,2025-06-17,1,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM32/Log BM3...
...,...,...,...,...,...,...
148,BM38,2025-10-09,1,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM38/2025_10...
149,BM38,2025-10-10,1,Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM38/2025_10...
150,BM38,2025-10-11,1,Rev Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM38/2025_10...
151,BM38,2025-10-12,1,Rev Set A,/groups/spruston/home/moharb/DELTA_Behavior/ou...,/nearline/spruston/Boaz/Mesoscope/BM38/2025_10...


In [5]:

# Aggregate to one row per day per animal
if sessions.empty:
    raise RuntimeError("No sessions found. Check HDF5_ROOT or file naming.")

daily = (
    sessions
    .groupby(["animal", "date"], as_index=False)
    .agg(
        cue_set=("cue_set", lambda x: ", ".join(sorted({c for c in x if pd.notna(c)})) or None),
        sessions=("session", "nunique"),
    )
    .sort_values(["animal", "date"])
)

daily


,animal,date,cue_set,sessions
0,BM32,2025-06-16,Set A,2
1,BM32,2025-06-17,Set A,2
2,BM32,2025-06-18,Set A,2
3,BM32,2025-06-19,Set A,2
4,BM32,2025-06-20,Set A,2
...,...,...,...,...
82,BM38,2025-10-09,Set A,1
83,BM38,2025-10-10,Set A,1
84,BM38,2025-10-11,Rev Set A,1
85,BM38,2025-10-12,Rev Set A,1


In [6]:

# Days missing a cue set (e.g., JSON file absent or missing cueSet tag)
missing_cue_days = daily[daily["cue_set"].isna()]
missing_cue_days


,animal,date,cue_set,sessions
49,BM35,2025-09-22,None,1


In [7]:

# Last three recorded days per animal
last_three = daily.groupby("animal").tail(4)
last_three


,animal,date,cue_set,sessions
9,BM32,2025-06-27,Set A,2
10,BM32,2025-06-28,Rev Set A,1
11,BM32,2025-06-29,Rev Set A,2
12,BM32,2025-06-30,Rev Set A,3
27,BM33,2025-08-02,Set A,1
28,BM33,2025-08-03,Rev Set A,2
29,BM33,2025-08-04,Rev Set A,2
30,BM33,2025-08-05,Rev Set A,2
41,BM34,2025-07-29,Set A,2
42,BM34,2025-07-30,Set A,3



If you see rows with `cue_set` as `NaN`, the HDF5 files do not contain the cue label, and the corresponding JSON logs are either missing or do not include a `cueSet` tag. In that case, the cue set cannot be recovered without the JSON.
